# Graph Neural Networks for Toxicology
### GCN · GAT · MPNN · GIN · AttentiveFP — All in One Script

**Author:** Himanshu Goel | [himanshugoel.github.io](https://himanshugoel.github.io)  
**Repo:** `computational-science-tutorials / gnn`

---

## Why molecules are graphs

A molecule is a natural graph:
- **Nodes** = atoms (with features: atomic number, charge, aromaticity, …)
- **Edges** = bonds (with features: bond type, conjugation, …)
- **Graph-level label** = toxicity endpoint (hERG / DILI / Ames / LD50)

Traditional ML uses fingerprints (fixed-size bit vectors). GNNs learn directly from the graph — no manual feature engineering needed.

```
Molecule graph                  GNN pipeline
─────────────────────           ────────────────────────────────────────
  C ── C ── C                   Atom features  →  Message passing  →
  |         |                                     Aggregate neighbors
  O ── N ── C                                     Update node states
                                                  Readout (sum/mean/att)
                                                  MLP head  →  prediction
```

## What this notebook covers

| Section | Content |
|---------|--------|
| 1. Setup | Install PyTorch Geometric, check GPU |
| 2. Molecule → Graph | Build atom/bond feature vectors from RDKit |
| 3. Dataset | Tox21 + hERG from MoleculeNet, scaffold split |
| 4. GCN | Graph Convolutional Network (Kipf 2017) |
| 5. GAT | Graph Attention Network (Veličković 2018) |
| 6. MPNN | Message Passing NN (Gilmer 2017) |
| 7. GIN | Graph Isomorphism Network (Xu 2019) |
| 8. AttentiveFP | Attention-based, SOTA on molecular tasks |
| 9. Benchmark | All 5 models head-to-head on same task |
| 10. Visualisation | Attention weights, UMAP of embeddings |
| 11. Uncertainty | MC Dropout, deep ensembles |
| 12. Tips & Pitfalls | What always goes wrong |

---
## Section 1 — Installation & Setup

In [ ]:
# ── Install (run once, then restart kernel) ───────────────────────────────────
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
# !pip install torch_geometric
# !pip install torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
# !pip install rdkit deepchem

# ── Colab one-liner ───────────────────────────────────────────────────────────
# !pip install torch torch_geometric rdkit deepchem

# ── Imports ───────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

# PyTorch Geometric
from torch_geometric.data import Data, Dataset, DataLoader
from torch_geometric.nn import (
    GCNConv, GATConv, GATv2Conv, NNConv, GINConv,
    global_mean_pool, global_add_pool, global_max_pool,
    Set2Set, AttentiveFP
)
from torch_geometric.utils import to_networkx

# RDKit
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, QED
from rdkit.Chem.Scaffolds import MurckoScaffold

# sklearn for evaluation
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.model_selection import StratifiedKFold

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
if device.type == 'cuda':
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Set seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

---
## Section 2 — Molecule → Graph Conversion

The foundation of all GNNs for molecules. We convert a SMILES string into a `torch_geometric.data.Data` object with:
- `x`          — node feature matrix  `[N_atoms, N_atom_features]`
- `edge_index` — edge connectivity    `[2, N_edges]`  (COO format)
- `edge_attr`  — edge feature matrix  `[N_edges, N_bond_features]`
- `y`          — graph-level label

In [ ]:
# ── 2.1 Atom feature vector ───────────────────────────────────────────────────
# 9 features per atom → 74-dimensional one-hot + numeric vector

ATOM_TYPES     = ['C','N','O','S','F','P','Cl','Br','I','other']
HYBRIDISATIONS = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
]

def one_hot(val, choices):
    """One-hot encode val from choices. Last bin = 'other'."""
    vec = [0] * len(choices)
    idx = choices.index(val) if val in choices else len(choices) - 1
    vec[idx] = 1
    return vec

def atom_features(atom) -> list:
    """
    Build a 74-dimensional feature vector for one atom.

    Features:
      [0:10]  atom type one-hot (C N O S F P Cl Br I other)
      [10:15] degree one-hot (0 1 2 3 4+)
      [15:20] formal charge one-hot (-1 -2 0 1 2)
      [20:25] num Hs one-hot (0 1 2 3 4)
      [25:30] hybridisation one-hot (sp sp2 sp3 sp3d sp3d2)
      [30]    is aromatic (0/1)
      [31]    is in ring (0/1)
      [32:38] ring size membership (3 4 5 6 7 8)
    """
    ring_info = atom.GetOwningMol().GetRingInfo()
    return (
        one_hot(atom.GetSymbol(),        ATOM_TYPES) +          # 10
        one_hot(atom.GetDegree(),        [0,1,2,3,4]) +          # 5
        one_hot(atom.GetFormalCharge(),  [-1,-2,0,1,2]) +        # 5
        one_hot(atom.GetTotalNumHs(),    [0,1,2,3,4]) +          # 5
        one_hot(atom.GetHybridization(), HYBRIDISATIONS) +       # 5
        [int(atom.GetIsAromatic())] +                            # 1
        [int(atom.IsInRing())] +                                 # 1
        [int(ring_info.IsAtomInRingOfSize(atom.GetIdx(), s))
         for s in [3,4,5,6,7,8]]                                  # 6
    )  # total: 38 features

N_ATOM_FEATURES = len(atom_features(
    Chem.MolFromSmiles('CC').GetAtomWithIdx(0)
))
print(f"Atom feature dimension: {N_ATOM_FEATURES}")

In [ ]:
# ── 2.2 Bond feature vector ───────────────────────────────────────────────────
BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]

def bond_features(bond) -> list:
    """
    Build a 10-dimensional feature vector for one bond.

    Features:
      [0:4]  bond type one-hot (single double triple aromatic)
      [4]    is conjugated
      [5]    is in ring
      [6:10] stereo one-hot (none any E Z)
    """
    stereo = [
        Chem.rdchem.BondStereo.STEREONONE,
        Chem.rdchem.BondStereo.STEREOANY,
        Chem.rdchem.BondStereo.STEREOE,
        Chem.rdchem.BondStereo.STEREOZ,
    ]
    return (
        one_hot(bond.GetBondType(), BOND_TYPES) +          # 4
        [int(bond.GetIsConjugated())] +                    # 1
        [int(bond.IsInRing())] +                           # 1
        one_hot(bond.GetStereo(), stereo)                   # 4
    )  # total: 10 features

N_BOND_FEATURES = len(bond_features(
    Chem.MolFromSmiles('CC').GetBondWithIdx(0)
))
print(f"Bond feature dimension: {N_BOND_FEATURES}")

In [ ]:
# ── 2.3 SMILES → PyG Data object ─────────────────────────────────────────────
def mol_to_graph(smiles: str, label=None) -> Data | None:
    """
    Convert a SMILES string to a PyTorch Geometric Data object.

    Returns:
        Data with:
          .x          float32 [N_atoms, N_atom_features]
          .edge_index long    [2, 2*N_bonds]  (both directions!)
          .edge_attr  float32 [2*N_bonds, N_bond_features]
          .y          float32 [1]  or None
          .smiles     str
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # ── Node features ──────────────────────────────────────────────────────────
    x = torch.tensor(
        [atom_features(a) for a in mol.GetAtoms()],
        dtype=torch.float32
    )  # [N_atoms, 38]

    # ── Edge index + edge features ─────────────────────────────────────────────
    # Bonds are undirected — add both (i→j) and (j→i)
    row, col, edge_feats = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        feat = bond_features(bond)
        row  += [i, j]
        col  += [j, i]
        edge_feats += [feat, feat]  # same features both directions

    edge_index = torch.tensor([row, col], dtype=torch.long)        # [2, 2*E]
    edge_attr  = torch.tensor(edge_feats, dtype=torch.float32)     # [2*E, 10]

    # ── Graph label ────────────────────────────────────────────────────────────
    y = None
    if label is not None:
        y = torch.tensor([label], dtype=torch.float32)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=y, smiles=smiles)


# ── Test it ────────────────────────────────────────────────────────────────────
aspirin_graph = mol_to_graph("CC(=O)Oc1ccccc1C(=O)O", label=0)
print(f"Aspirin graph:")
print(f"  x (atom features) : {aspirin_graph.x.shape}  → {aspirin_graph.num_nodes} atoms × {N_ATOM_FEATURES} features")
print(f"  edge_index         : {aspirin_graph.edge_index.shape}  → {aspirin_graph.num_edges} directed edges")
print(f"  edge_attr          : {aspirin_graph.edge_attr.shape}  → each bond has {N_BOND_FEATURES} features")
print(f"  y (label)          : {aspirin_graph.y}")
print(f"  is_undirected      : {aspirin_graph.is_undirected()}")

In [ ]:
# ── 2.4 Visualise the graph structure ─────────────────────────────────────────
import networkx as nx
from rdkit.Chem import Draw

def visualise_mol_graph(smiles, title=""):
    """Side-by-side: RDKit 2D structure + NetworkX graph."""
    mol   = Chem.MolFromSmiles(smiles)
    graph = mol_to_graph(smiles)
    G     = to_networkx(graph, to_undirected=True)

    # Atom labels
    labels = {i: mol.GetAtomWithIdx(i).GetSymbol() for i in range(mol.GetNumAtoms())}
    # Colour by element
    colour_map = {'C':'#A0A0A0','N':'#4444FF','O':'#FF4444','S':'#DDDD00',
                  'F':'#44FF44','Cl':'#00CCCC','Br':'#CC7722','default':'#888888'}
    node_colors = [
        colour_map.get(mol.GetAtomWithIdx(n).GetSymbol(), colour_map['default'])
        for n in G.nodes()
    ]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left: RDKit 2D
    mol_img = Draw.MolToImage(mol, size=(450, 350))
    axes[0].imshow(mol_img); axes[0].axis('off')
    axes[0].set_title(f"{title} — 2D structure", fontsize=13, fontweight='bold')

    # Right: NetworkX graph
    pos = nx.spring_layout(G, seed=42)
    nx.draw_networkx(
        G, pos, labels=labels, node_color=node_colors,
        node_size=800, font_size=11, font_weight='bold',
        edge_color='#555555', width=2.0, ax=axes[1]
    )
    axes[1].set_title(
        f"{title} — Graph representation\n"
        f"{graph.num_nodes} nodes · {graph.num_edges//2} edges · "
        f"{N_ATOM_FEATURES}d node feats · {N_BOND_FEATURES}d edge feats",
        fontsize=11
    )
    axes[1].axis('off')
    plt.tight_layout(); plt.show()

visualise_mol_graph("CC(=O)Oc1ccccc1C(=O)O", "Aspirin")
visualise_mol_graph("Cn1cnc2c1c(=O)n(C)c(=O)n2C",  "Caffeine")

---
## Section 3 — Dataset: Tox21 (hERG subset)

We use a curated hERG cardiotoxicity dataset — a standard benchmark for molecular GNNs.

**hERG (IKr):** The hERG potassium channel. Blocking it prolongs QT interval → cardiac arrhythmia → death. Every drug candidate must be screened.

In [ ]:
# ── 3.1 Load or simulate a molecular toxicity dataset ─────────────────────────
# In practice: use DeepChem or MoleculeNet
#   import deepchem as dc
#   tasks, datasets, transformers = dc.molnet.load_herg_central(featurizer='GraphConv')

# We simulate a realistic hERG dataset here so the notebook runs without downloads.
# Replace this block with real data for production use.

def simulate_tox_dataset(n=500, seed=42):
    """
    Generate a realistic synthetic hERG dataset.
    Activity is correlated with MW, LogP, and TPSA — as in real data.
    """
    np.random.seed(seed)
    templates = [
        "CCN(CC)CCOc1ccc(CC2CCN(C)CC2)cc1",             # typical hERG actives
        "c1ccc2c(c1)CCN2CCc1ccc(F)cc1",
        "O=C(NCCCN1CCOCC1)c1ccc2c(c1)CCO2",
        "CC(C)Cc1ccc(CC2CCNCC2)cc1",
        "CC(=O)Oc1ccccc1C(=O)O",                         # inactives
        "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
        "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
        "CC(=O)Nc1ccc(O)cc1",
        "NCCc1ccc(O)c(O)c1",
        "OC(=O)c1ccccc1",
        "c1ccc2ccccc2c1",
        "c1ccncc1",
    ]
    smiles_list, labels = [], []
    for i in range(n):
        smi = templates[i % len(templates)]
        mol = Chem.MolFromSmiles(smi)
        if mol:
            # Realistic hERG activity: high LogP + basic N + high MW → active
            logp = Descriptors.MolLogP(mol)
            mw   = Descriptors.MolWt(mol)
            tpsa = Descriptors.TPSA(mol)
            score = 0.35*max(0,logp-2) + 0.002*max(0,mw-300) - 0.008*tpsa
            prob  = 1 / (1 + np.exp(-score + np.random.normal(0, 0.5)))
            label = int(prob > 0.45)
            smiles_list.append(smi)
            labels.append(label)

    print(f"Dataset: {len(smiles_list)} compounds")
    print(f"  Actives (hERG blockers) : {sum(labels)}  ({sum(labels)/len(labels)*100:.0f}%)")
    print(f"  Inactives               : {len(labels)-sum(labels)}  ({(1-sum(labels)/len(labels))*100:.0f}%)")
    return smiles_list, labels

smiles_list, labels = simulate_tox_dataset(n=600)

In [ ]:
# ── 3.2 Convert all SMILES to graphs ──────────────────────────────────────────
graphs = []
for smi, lbl in zip(smiles_list, labels):
    g = mol_to_graph(smi, label=lbl)
    if g is not None:
        graphs.append(g)

print(f"Valid graphs: {len(graphs)}")
print(f"Example graph: {graphs[0]}")
print(f"  nodes={graphs[0].num_nodes}, edges={graphs[0].num_edges}, y={graphs[0].y.item()}")

In [ ]:
# ── 3.3 Scaffold split ────────────────────────────────────────────────────────
# Critical! Random split leaks information across structurally similar molecules.

def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))

def scaffold_split(graphs, test_frac=0.15, val_frac=0.15, seed=42):
    """Split graphs by Murcko scaffold. No scaffold appears in multiple splits."""
    np.random.seed(seed)
    scaffold_to_idx = defaultdict(list)
    for i, g in enumerate(graphs):
        s = get_scaffold(g.smiles)
        scaffold_to_idx[s if s else f"__noscore_{i}"].append(i)

    # Sort by frequency (rare scaffolds go to test)
    scaffold_sets = sorted(scaffold_to_idx.values(), key=len)

    n       = len(graphs)
    n_test  = int(n * test_frac)
    n_val   = int(n * val_frac)

    train_idx, val_idx, test_idx = [], [], []
    for indices in scaffold_sets:
        if len(test_idx) < n_test:
            test_idx.extend(indices)
        elif len(val_idx) < n_val:
            val_idx.extend(indices)
        else:
            train_idx.extend(indices)

    return (
        [graphs[i] for i in train_idx],
        [graphs[i] for i in val_idx],
        [graphs[i] for i in test_idx],
    )

train_data, val_data, test_data = scaffold_split(graphs)
print(f"Split: train={len(train_data)}  val={len(val_data)}  test={len(test_data)}")

BATCH = 32
train_loader = DataLoader(train_data, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH)
test_loader  = DataLoader(test_data,  batch_size=BATCH)

---
## Section 4 — GCN: Graph Convolutional Network

**Kipf & Welling (2017)** — the simplest and most popular GNN.

### How it works
Each node aggregates the **mean** of its neighbour features, weighted by graph structure:

$$H^{(l+1)} = \sigma\left(\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2} H^{(l)} W^{(l)}\right)$$

where $\tilde{A} = A + I$ (add self-loops) and $\tilde{D}$ is the degree matrix.

**Limitation:** All neighbours get equal weight. Edge features are ignored.

In [ ]:
# ── 4.1 GCN model ────────────────────────────────────────────────────────────
class GCN(nn.Module):
    """
    3-layer GCN for binary graph classification.

    Architecture:
      Input features (38)
        → GCNConv(38 → 64)  + BatchNorm + ReLU
        → GCNConv(64 → 128) + BatchNorm + ReLU
        → GCNConv(128 → 128)+ BatchNorm + ReLU
        → GlobalMeanPool  → [batch, 128]
        → Linear(128 → 64) + ReLU + Dropout(0.3)
        → Linear(64 → 1)   → logit
    """
    def __init__(self, in_channels=N_ATOM_FEATURES, hidden=64, dropout=0.3):
        super().__init__()
        # Graph convolution layers
        self.conv1 = GCNConv(in_channels, hidden)
        self.conv2 = GCNConv(hidden, hidden * 2)
        self.conv3 = GCNConv(hidden * 2, hidden * 2)
        # Batch normalisation stabilises training
        self.bn1   = nn.BatchNorm1d(hidden)
        self.bn2   = nn.BatchNorm1d(hidden * 2)
        self.bn3   = nn.BatchNorm1d(hidden * 2)
        # Classifier head
        self.fc1     = nn.Linear(hidden * 2, hidden)
        self.fc2     = nn.Linear(hidden, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # 3 rounds of message passing
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.relu(self.bn3(self.conv3(x, edge_index)))

        # Readout: mean pool all atoms → one vector per molecule
        x = global_mean_pool(x, batch)   # [n_graphs, 128]

        # MLP head
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(-1)   # logit [n_graphs]

    def get_embedding(self, data):
        """Return graph-level embedding (before final FC)."""
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.relu(self.bn3(self.conv3(x, edge_index)))
        return global_mean_pool(x, batch)


# Inspect the model
gcn = GCN().to(device)
n_params = sum(p.numel() for p in gcn.parameters() if p.requires_grad)
print(f"GCN parameters: {n_params:,}")
print(gcn)

In [ ]:
# ── 4.2 Training loop (reusable for all models) ────────────────────────────────
def train_epoch(model, loader, optimiser, criterion):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimiser.zero_grad()
        logits = model(batch)
        loss   = criterion(logits, batch.y.squeeze())
        loss.backward()
        optimiser.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(batch.y.cpu().numpy().astype(int))
    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    auc    = roc_auc_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.5
    auprc  = average_precision_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.5
    return {'auc': auc, 'auprc': auprc, 'preds': y_pred, 'labels': y_true}


def train_model(model, name, epochs=60, lr=3e-4, patience=15):
    """Train with early stopping on validation AUC."""
    model = model.to(device)
    optimiser = Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = CosineAnnealingLR(optimiser, T_max=epochs)
    # Weighted BCE for class imbalance
    pos_weight = torch.tensor([sum(1-l for l in labels) / max(sum(labels), 1)]).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_val_auc, best_weights, patience_count = 0, None, 0
    history = {'train_loss': [], 'val_auc': []}

    for epoch in range(1, epochs + 1):
        loss     = train_epoch(model, train_loader, optimiser, criterion)
        val_res  = evaluate(model, val_loader)
        scheduler.step()
        history['train_loss'].append(loss)
        history['val_auc'].append(val_res['auc'])

        if val_res['auc'] > best_val_auc:
            best_val_auc = val_res['auc']
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1

        if epoch % 10 == 0 or epoch == 1:
            print(f"  [{name}] Epoch {epoch:3d} | loss={loss:.4f} | val AUC={val_res['auc']:.4f}")

        if patience_count >= patience:
            print(f"  Early stop at epoch {epoch}")
            break

    # Restore best weights
    model.load_state_dict({k: v.to(device) for k, v in best_weights.items()})
    test_res = evaluate(model, test_loader)
    print(f"\n  [{name}] Test AUC={test_res['auc']:.4f}  AUPRC={test_res['auprc']:.4f}")
    return model, history, test_res


# Train GCN
print("Training GCN...")
gcn, gcn_history, gcn_test = train_model(GCN(), 'GCN')

---
## Section 5 — GAT: Graph Attention Network

**Veličković et al. (2018)** — adds **learnable attention weights** between neighbours.

Instead of equal-weight averaging, each edge gets a scalar attention coefficient:

$$\alpha_{ij} = \frac{\exp\left(\text{LeakyReLU}(\mathbf{a}^\top [\mathbf{W}h_i \| \mathbf{W}h_j])\right)}{\sum_{k\in\mathcal{N}(i)} \exp\left(\text{LeakyReLU}(\mathbf{a}^\top [\mathbf{W}h_i \| \mathbf{W}h_k])\right)}$$

Multi-head attention (`heads=8`) runs $K$ independent attention mechanisms and concatenates (or averages) their outputs → more stable training.

In [ ]:
# ── 5.1 GAT model ─────────────────────────────────────────────────────────────
class GAT(nn.Module):
    """
    Graph Attention Network with multi-head attention.
    Uses GATv2Conv — more expressive than original GAT
    (Brody, Alon & Yahav, 2022).

    Layer 1: 8 attention heads × 16 dims = 128 output dims (concat)
    Layer 2: 4 attention heads × 32 dims = 128 output dims (concat)
    Layer 3: 1 attention head  × 128 dims (average, final readout)
    """
    def __init__(self, in_channels=N_ATOM_FEATURES, hidden=128, heads1=8, heads2=4, dropout=0.3):
        super().__init__()
        # Layer 1: 8 heads × 16 = 128 dim output
        self.conv1 = GATv2Conv(in_channels, hidden // heads1,
                               heads=heads1, dropout=dropout, concat=True)
        # Layer 2: 4 heads × 32 = 128 dim output
        self.conv2 = GATv2Conv(hidden, hidden // heads2,
                               heads=heads2, dropout=dropout, concat=True)
        # Layer 3: 1 head × 128 = 128 dim output (averaged)
        self.conv3 = GATv2Conv(hidden, hidden,
                               heads=1, dropout=dropout, concat=False)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.bn3   = nn.BatchNorm1d(hidden)
        self.fc1     = nn.Linear(hidden, hidden // 2)
        self.fc2     = nn.Linear(hidden // 2, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, data, return_attention=False):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = F.elu(self.bn1(self.conv1(x, edge_index)))
        x = F.elu(self.bn2(self.conv2(x, edge_index)))

        if return_attention:
            x, (edge_idx, attn_weights) = self.conv3(
                x, edge_index, return_attention_weights=True)
            x = F.elu(self.bn3(x))
            x = global_mean_pool(x, batch)
            x = self.dropout(F.relu(self.fc1(x)))
            return self.fc2(x).squeeze(-1), (edge_idx, attn_weights)

        x = F.elu(self.bn3(self.conv3(x, edge_index)))
        x = global_mean_pool(x, batch)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(-1)

gat = GAT().to(device)
print(f"GAT parameters: {sum(p.numel() for p in gat.parameters()):,}")
print("Training GAT...")
gat, gat_history, gat_test = train_model(GAT(), 'GAT')

---
## Section 6 — MPNN: Message Passing Neural Network

**Gilmer et al. (2017)** — the theoretical framework underlying all GNNs. Key innovation: **uses bond features** (edge attributes) in the message function.

### Message passing phases:
1. **Message:** $m_{ij}^{(t)} = M_t(h_i^{(t)}, h_j^{(t)}, e_{ij})$  — learns what to pass based on both atoms AND the bond between them
2. **Aggregate:** $m_i^{(t)} = \sum_{j \in N(i)} m_{ij}^{(t)}$
3. **Update:** $h_i^{(t+1)} = U_t(h_i^{(t)}, m_i^{(t)})$  — uses GRU cell
4. **Readout:** $\hat{y} = R(\{h_i^{(T)} | i \in G\})$  — Set2Set attention

In [ ]:
# ── 6.1 MPNN model ────────────────────────────────────────────────────────────
class MPNN(nn.Module):
    """
    Message Passing NN with:
      - NNConv (edge-conditioned convolution) for message function
      - GRU cell for update function
      - Set2Set for readout (permutation-invariant attention pooling)

    NNConv: the message network is itself a small MLP parameterised
    by the edge features. So the aggregated message depends on the
    bond type between atoms — crucial for chemistry!
    """
    def __init__(self, in_channels=N_ATOM_FEATURES, edge_dim=N_BOND_FEATURES,
                 hidden=64, steps=3, dropout=0.3):
        super().__init__()
        self.hidden = hidden
        self.steps  = steps

        # Project atom features to hidden dim
        self.atom_proj = nn.Linear(in_channels, hidden)

        # Edge MLP: maps bond features → hidden × hidden weight matrix
        # NNConv uses this to compute per-edge message functions
        edge_nn = nn.Sequential(
            nn.Linear(edge_dim, hidden * 4),
            nn.ReLU(),
            nn.Linear(hidden * 4, hidden * hidden),  # output: weight matrix
        )
        self.conv = NNConv(hidden, hidden, edge_nn, aggr='add')

        # GRU update: update atom state using aggregated messages
        self.gru = nn.GRUCell(hidden, hidden)

        # Set2Set readout: attention-based permutation-invariant pooling
        # Returns 2×hidden because it concatenates query and value
        self.set2set = Set2Set(hidden, processing_steps=3)

        # Classifier head
        self.fc1     = nn.Linear(hidden * 2, hidden)   # Set2Set → 2×hidden
        self.fc2     = nn.Linear(hidden, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, data):
        x, edge_index, edge_attr, batch = (
            data.x, data.edge_index, data.edge_attr, data.batch
        )

        # Initial projection
        h = F.relu(self.atom_proj(x))  # [N, hidden]

        # T rounds of message passing with GRU update
        for _ in range(self.steps):
            m = F.relu(self.conv(h, edge_index, edge_attr))  # aggregate messages
            h = self.gru(m, h)                                # GRU update

        # Readout: Set2Set attention pooling
        out = self.set2set(h, batch)          # [n_graphs, 2*hidden]
        out = self.dropout(F.relu(self.fc1(out)))
        return self.fc2(out).squeeze(-1)


mpnn = MPNN().to(device)
print(f"MPNN parameters: {sum(p.numel() for p in mpnn.parameters()):,}")
print("Training MPNN...")
mpnn, mpnn_history, mpnn_test = train_model(MPNN(), 'MPNN')

---
## Section 7 — GIN: Graph Isomorphism Network

**Xu et al. (2019)** — theoretically the most expressive simple GNN. Proved to be as powerful as the Weisfeiler-Lehman (WL) graph isomorphism test.

Key idea: use **SUM aggregation** (not mean), and add a learnable scalar $\epsilon$ for the self-loop:

$$h_i^{(l+1)} = \text{MLP}\left((1 + \epsilon^{(l)}) h_i^{(l)} + \sum_{j \in \mathcal{N}(i)} h_j^{(l)}\right)$$

Mean pooling can't distinguish certain graph structures that sum pooling can.

In [ ]:
# ── 7.1 GIN model ─────────────────────────────────────────────────────────────
class GIN(nn.Module):
    """
    Graph Isomorphism Network.

    Each GINConv layer:
      h_i' = MLP( (1 + eps) * h_i + SUM_{j in N(i)} h_j )
      eps is learned (train_eps=True)

    Concatenated readout (JK — jumping knowledge):
      pool each layer separately, concat → richer representation
    """
    def __init__(self, in_channels=N_ATOM_FEATURES, hidden=64, n_layers=5, dropout=0.3):
        super().__init__()
        self.n_layers = n_layers
        self.dropout  = nn.Dropout(dropout)
        self.convs    = nn.ModuleList()
        self.bns      = nn.ModuleList()

        for i in range(n_layers):
            in_dim  = in_channels if i == 0 else hidden
            # Each GIN layer has a 2-layer MLP inside
            mlp = nn.Sequential(
                nn.Linear(in_dim, hidden * 2),
                nn.BatchNorm1d(hidden * 2),
                nn.ReLU(),
                nn.Linear(hidden * 2, hidden),
            )
            self.convs.append(GINConv(mlp, train_eps=True))  # learnable epsilon
            self.bns.append(nn.BatchNorm1d(hidden))

        # JK (Jumping Knowledge): concatenate embeddings from each layer
        # → richer representation than just the last layer
        self.fc1 = nn.Linear(hidden * n_layers, hidden)
        self.fc2 = nn.Linear(hidden, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        pooled_per_layer = []
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, edge_index)))
            x = self.dropout(x)
            # Pool each layer separately (JK connection)
            pooled_per_layer.append(global_add_pool(x, batch))

        # Concatenate all layers (JK — uses info from every depth)
        out = torch.cat(pooled_per_layer, dim=-1)      # [n_graphs, hidden*n_layers]
        out = self.dropout(F.relu(self.fc1(out)))
        return self.fc2(out).squeeze(-1)


gin = GIN().to(device)
print(f"GIN parameters: {sum(p.numel() for p in gin.parameters()):,}")
print("Training GIN...")
gin, gin_history, gin_test = train_model(GIN(), 'GIN')

---
## Section 8 — AttentiveFP

**Xiong et al. (2020, JCIM)** — SOTA on many molecular tasks. Combines:
1. **Atom-level attention** — neighbours vote on each atom's update
2. **Graph-level attention** — atoms vote on the molecular fingerprint

Specifically designed for molecular property prediction — uses all atom and bond features naturally.

In [ ]:
# ── 8.1 AttentiveFP model ─────────────────────────────────────────────────────
# PyG provides AttentiveFP directly — we just wrap it
class AttentiveFPModel(nn.Module):
    """
    Attentive FP (Xiong 2020):
      - Graph attention at atom level (atom attention)
      - Graph attention at molecule level (super-atom/readout attention)
      - Dropout regularisation
    """
    def __init__(self, in_channels=N_ATOM_FEATURES, edge_dim=N_BOND_FEATURES,
                 hidden=200, num_layers=2, num_timesteps=2, dropout=0.2):
        super().__init__()
        self.afp = AttentiveFP(
            in_channels=in_channels,
            hidden_channels=hidden,
            out_channels=1,
            edge_dim=edge_dim,
            num_layers=num_layers,       # atom-level attention rounds
            num_timesteps=num_timesteps, # graph-level attention rounds
            dropout=dropout,
        )

    def forward(self, data):
        out = self.afp(data.x, data.edge_index, data.edge_attr, data.batch)
        return out.squeeze(-1)


afp = AttentiveFPModel().to(device)
print(f"AttentiveFP parameters: {sum(p.numel() for p in afp.parameters()):,}")
print("Training AttentiveFP...")
afp, afp_history, afp_test = train_model(AttentiveFPModel(), 'AttentiveFP', lr=1e-4)

---
## Section 9 — Benchmark Comparison

In [ ]:
# ── 9.1 All models head-to-head ───────────────────────────────────────────────
results = {
    'GCN':         gcn_test,
    'GAT':         gat_test,
    'MPNN':        mpnn_test,
    'GIN':         gin_test,
    'AttentiveFP': afp_test,
}
histories = {
    'GCN':         gcn_history,
    'GAT':         gat_history,
    'MPNN':        mpnn_history,
    'GIN':         gin_history,
    'AttentiveFP': afp_history,
}

# ── Summary table ──────────────────────────────────────────────────────────────
model_info = {
    'GCN':         {'params': sum(p.numel() for p in gcn.parameters()),  'edge_feats': False},
    'GAT':         {'params': sum(p.numel() for p in gat.parameters()),  'edge_feats': False},
    'MPNN':        {'params': sum(p.numel() for p in mpnn.parameters()), 'edge_feats': True},
    'GIN':         {'params': sum(p.numel() for p in gin.parameters()),  'edge_feats': False},
    'AttentiveFP': {'params': sum(p.numel() for p in afp.parameters()),  'edge_feats': True},
}

print(f"{'Model':15s} {'Test AUC':>10} {'AUPRC':>8} {'Params':>10} {'Bond feats':>12}")
print("-" * 60)
for name, res in results.items():
    info = model_info[name]
    print(f"{name:15s} {res['auc']:>10.4f} {res['auprc']:>8.4f} "
          f"{info['params']:>10,} {'Yes' if info['edge_feats'] else 'No':>12}")

best = max(results, key=lambda k: results[k]['auc'])
print(f"\nBest model: {best} (AUC={results[best]['auc']:.4f})")

In [ ]:
# ── 9.2 Comprehensive benchmark plot ──────────────────────────────────────────
fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

COLOURS = {'GCN':'#1565C0','GAT':'#E74C3C','MPNN':'#27AE60',
           'GIN':'#E67E22','AttentiveFP':'#8E44AD'}

# ── Panel 1: Training curves ───────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
for name, hist in histories.items():
    ax1.plot(hist['val_auc'], label=name, color=COLOURS[name], lw=2.2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Validation AUC')
ax1.set_title('Training Curves — Validation AUC', fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.35)
ax1.set_ylim([0.45, 1.02])

# ── Panel 2: ROC curves ───────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
for name, res in results.items():
    fpr, tpr, _ = roc_curve(res['labels'], res['preds'])
    ax2.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})",
             color=COLOURS[name], lw=2.2)
ax2.plot([0,1],[0,1],'k--',lw=1.0,alpha=0.5)
ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curves (Test Set)', fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.35)

# ── Panel 3: AUC bar chart ─────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
names_list = list(results.keys())
aucs       = [results[n]['auc']   for n in names_list]
auprcs     = [results[n]['auprc'] for n in names_list]
x          = np.arange(len(names_list))
b1 = ax3.bar(x - 0.2, aucs,   0.35, label='ROC-AUC', color=[COLOURS[n] for n in names_list], alpha=0.9)
b2 = ax3.bar(x + 0.2, auprcs, 0.35, label='PR-AUC',  color=[COLOURS[n] for n in names_list], alpha=0.5)
ax3.set_xticks(x); ax3.set_xticklabels(names_list, rotation=20, fontsize=9)
ax3.set_ylabel('Score'); ax3.set_ylim([0.4, 1.05])
ax3.set_title('Test AUC & PR-AUC Comparison', fontweight='bold')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.35, axis='y')
for bar, val in zip(b1, aucs):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{val:.3f}', ha='center', fontsize=8.5, fontweight='bold')

# ── Panel 4: Loss curves ───────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
for name, hist in histories.items():
    ax4.plot(hist['train_loss'], label=name, color=COLOURS[name], lw=2.0)
ax4.set_xlabel('Epoch'); ax4.set_ylabel('Train Loss (BCE)')
ax4.set_title('Training Loss', fontweight='bold')
ax4.legend(fontsize=9); ax4.grid(True, alpha=0.35)

# ── Panel 5: Param count vs AUC ───────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
for name in names_list:
    ax5.scatter(model_info[name]['params'] / 1000, results[name]['auc'],
                s=200, color=COLOURS[name], zorder=5, label=name,
                marker='*' if model_info[name]['edge_feats'] else 'o')
    ax5.annotate(name, xy=(model_info[name]['params']/1000, results[name]['auc']),
                 xytext=(6, 4), textcoords='offset points', fontsize=9)
ax5.set_xlabel('Parameters (×1000)')
ax5.set_ylabel('Test AUC')
ax5.set_title('Efficiency: Params vs AUC\n(★ = uses bond features)', fontweight='bold')
ax5.grid(True, alpha=0.35)

# ── Panel 6: Architecture summary table ───────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
arch_data = [
    ['Model',        'Aggregation', 'Bond feats', 'Update', 'Readout',    'Key paper'],
    ['GCN',          'Mean (norm)', 'No',         'Linear', 'Mean pool',  'Kipf 2017'],
    ['GAT',          'Attention',   'No',         'Linear', 'Mean pool',  'Veličković 2018'],
    ['MPNN',         'Sum+NNConv',  'Yes',        'GRU',    'Set2Set',    'Gilmer 2017'],
    ['GIN',          'Sum (MLP)',   'No',         'MLP',    'Sum+JK',     'Xu 2019'],
    ['AttentiveFP',  'GRU+Attn',   'Yes',        'GRU',    'Attn pool',  'Xiong 2020'],
]
table = ax6.table(cellText=arch_data[1:], colLabels=arch_data[0],
                  cellLoc='center', loc='center', bbox=[0,0,1,1])
table.auto_set_font_size(False); table.set_fontsize(9)
for j in range(len(arch_data[0])):
    table[0,j].set_facecolor('#1565C0')
    table[0,j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(arch_data)):
    for j in range(len(arch_data[0])):
        table[i,j].set_facecolor('#EEF5FF' if i%2==0 else 'white')
ax6.set_title('GNN Architecture Summary', fontweight='bold', pad=12)

fig.suptitle('GNN for Molecular Toxicology — Complete Benchmark\nhERG Cardiotoxicity (Scaffold Split)',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('gnn_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: gnn_benchmark.png")

---
## Section 10 — Visualisation: Attention Weights & UMAP

In [ ]:
# ── 10.1 GAT attention weight visualisation ───────────────────────────────────
# Which bonds does the model 'attend to' for a specific molecule?

def visualise_attention(model, smiles, title=""):
    """Overlay GAT attention weights on the molecular graph."""
    mol   = Chem.MolFromSmiles(smiles)
    graph = mol_to_graph(smiles)
    if graph is None: return

    # Get attention weights from GAT
    model.eval()
    batch = graph.to(device)
    batch.batch = torch.zeros(batch.num_nodes, dtype=torch.long, device=device)
    with torch.no_grad():
        _, (edge_idx, attn) = model(batch, return_attention=True)

    edge_idx = edge_idx.cpu().numpy().T  # [E, 2]
    attn     = attn.cpu().numpy().flatten()

    # Build edge attention dict (average over duplicate directed edges)
    bond_attn = defaultdict(list)
    for (i, j), w in zip(edge_idx, attn):
        bond_attn[(min(i,j), max(i,j))].append(w)
    bond_attn = {k: np.mean(v) for k, v in bond_attn.items()}

    G   = to_networkx(graph.cpu(), to_undirected=True)
    pos = nx.spring_layout(G, seed=42)

    # Node labels
    node_labels = {i: mol.GetAtomWithIdx(i).GetSymbol() for i in G.nodes()}

    # Edge widths and colours from attention
    max_a = max(bond_attn.values()) if bond_attn else 1
    edge_widths  = [bond_attn.get((min(u,v),max(u,v)), 0) / max_a * 10 + 0.5 for u,v in G.edges()]
    edge_colours = [plt.cm.Reds(bond_attn.get((min(u,v),max(u,v)), 0) / max_a) for u,v in G.edges()]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 2D structure
    from rdkit.Chem import Draw
    img = Draw.MolToImage(mol, size=(400, 320))
    axes[0].imshow(img); axes[0].axis('off')
    axes[0].set_title(f"{title} — 2D structure")

    # Attention graph
    nx.draw_networkx_nodes(G, pos, node_color='#B8D4F5', node_size=700, ax=axes[1])
    nx.draw_networkx_labels(G, pos, node_labels, font_size=11, font_weight='bold', ax=axes[1])
    nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color=edge_colours, ax=axes[1])
    axes[1].set_title(f"{title} — GAT attention\n(thicker/redder = higher attention)", fontsize=11)
    axes[1].axis('off')

    # Colourbar
    sm = plt.cm.ScalarMappable(cmap='Reds')
    sm.set_clim(0, max_a)
    plt.colorbar(sm, ax=axes[1], label='Attention weight')
    plt.tight_layout(); plt.show()

visualise_attention(gat, "CCN(CC)CCOc1ccc(CC2CCN(C)CC2)cc1", title="hERG active")
visualise_attention(gat, "CC(=O)Oc1ccccc1C(=O)O", title="Aspirin (hERG inactive)")

In [ ]:
# ── 10.2 UMAP of GNN embeddings ───────────────────────────────────────────────
# Do the learned embeddings separate hERG actives from inactives?

# pip install umap-learn
try:
    from umap import UMAP
    umap_available = True
except ImportError:
    from sklearn.manifold import TSNE
    umap_available = False
    print("umap-learn not found — using t-SNE instead. pip install umap-learn")

@torch.no_grad()
def get_embeddings(model, loader):
    model.eval()
    embeds, labels_list = [], []
    for batch in loader:
        batch = batch.to(device)
        emb   = model.get_embedding(batch).cpu().numpy()
        embeds.append(emb)
        labels_list.append(batch.y.cpu().numpy())
    return np.vstack(embeds), np.concatenate(labels_list)

# Use GCN (has get_embedding method)
all_loader = DataLoader(graphs, batch_size=64)
embeddings, true_labels = get_embeddings(gcn, all_loader)

# Reduce to 2D
if umap_available:
    reducer = UMAP(n_components=2, n_neighbors=20, min_dist=0.3, random_state=42)
    coords  = reducer.fit_transform(embeddings)
    method  = 'UMAP'
else:
    reducer = TSNE(n_components=2, random_state=42, perplexity=20)
    coords  = reducer.fit_transform(embeddings)
    method  = 't-SNE'

fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(
    coords[true_labels==0, 0], coords[true_labels==0, 1],
    c='#1565C0', s=25, alpha=0.65, label='hERG inactive'
)
ax.scatter(
    coords[true_labels==1, 0], coords[true_labels==1, 1],
    c='#E74C3C', s=25, alpha=0.65, label='hERG active'
)
ax.set_title(f'{method} of GCN Graph Embeddings\nhERG Cardiotoxicity',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlabel(f'{method} 1'); ax.set_ylabel(f'{method} 2')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## Section 11 — Uncertainty Quantification

In drug discovery, knowing **how confident** the model is matters as much as the prediction itself. Overconfident models advance bad compounds.

Two standard methods:
1. **MC Dropout** — keep dropout active at inference, run N forward passes → variance = uncertainty
2. **Deep ensembles** — train K independent models, variance over predictions

In [ ]:
# ── 11.1 MC Dropout uncertainty ───────────────────────────────────────────────
@torch.no_grad()
def mc_dropout_predict(model, loader, n_passes=30):
    """
    Run N stochastic forward passes with dropout active.
    Returns mean prediction and uncertainty (std of predictions).
    """
    model.train()   # KEEP DROPOUT ON during inference!
    all_preds = []

    for _ in range(n_passes):
        pass_preds = []
        for batch in loader:
            batch  = batch.to(device)
            logits = model(batch)
            probs  = torch.sigmoid(logits).cpu().numpy()
            pass_preds.extend(probs)
        all_preds.append(pass_preds)

    all_preds = np.array(all_preds)     # [n_passes, n_molecules]
    mean_pred = all_preds.mean(axis=0)
    std_pred  = all_preds.std(axis=0)
    return mean_pred, std_pred

mean_preds, uncertainty = mc_dropout_predict(gcn, test_loader, n_passes=30)
true_labels_test        = np.array([g.y.item() for g in test_data])

# Calibration: are high-uncertainty predictions more often wrong?
wrong = (mean_preds > 0.5).astype(int) != true_labels_test
low_u  = uncertainty < np.percentile(uncertainty, 50)
high_u = ~low_u

print("MC Dropout Uncertainty Analysis (GCN):")
print(f"  Mean uncertainty:          {uncertainty.mean():.4f}")
print(f"  Error rate (low uncertainty):  {wrong[low_u].mean():.3f}   ← model is confident and correct")
print(f"  Error rate (high uncertainty): {wrong[high_u].mean():.3f}   ← model is uncertain and wrong more often")
print()
print("Rule: flag compounds with uncertainty > threshold for further assay, not just rejection.")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(mean_preds, uncertainty, c=['#E74C3C' if l else '#1565C0' for l in true_labels_test],
                s=30, alpha=0.6)
axes[0].axhline(np.percentile(uncertainty, 75), color='k', linestyle='--', lw=1.5,
               label='75th pct threshold')
axes[0].set_xlabel('Predicted probability (mean)'); axes[0].set_ylabel('Uncertainty (std)')
axes[0].set_title('Prediction vs Uncertainty\n(red=active, blue=inactive)', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

axes[1].hist(uncertainty[true_labels_test==0], bins=20, alpha=0.7, label='hERG inactive', color='#1565C0')
axes[1].hist(uncertainty[true_labels_test==1], bins=20, alpha=0.7, label='hERG active',   color='#E74C3C')
axes[1].set_xlabel('MC Dropout Uncertainty (std)'); axes[1].set_ylabel('Count')
axes[1].set_title('Uncertainty Distribution by True Class', fontweight='bold')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## Section 12 — Tips, Tricks & Common Pitfalls

In [ ]:
pitfalls = """
╔══════════════════════════════════════════════════════════════════════════╗
║          GNN for Molecules — Tips, Tricks & Pitfalls                    ║
╠══════════════════════════════════════════════════════════════════════════╣
║ DATA PREPARATION                                                         ║
║ ✓ Use scaffold split — NEVER random split for molecules                  ║
║   Random split: train/test share similar scaffolds → inflated AUC       ║
║ ✓ Standardize SMILES before graph conversion (salt stripping, etc.)     ║
║ ✓ Handle class imbalance: use pos_weight in BCEWithLogitsLoss           ║
║   pos_weight = n_negatives / n_positives                                 ║
║ ✓ Include BOTH directed edges (i→j and j→i) — GNN needs both           ║
╠══════════════════════════════════════════════════════════════════════════╣
║ MODEL DESIGN                                                             ║
║ ✓ 3–5 GNN layers is usually optimal. More → oversmoothing               ║
║   Oversmoothing: all nodes converge to same embedding after many layers  ║
║ ✓ Use BatchNorm between GNN layers — critical for stability             ║
║ ✓ Residual connections help deeper models (skip connections)            ║
║ ✓ Add edge features if available — MPNN/AttentiveFP > GCN/GAT on chem  ║
║ ✓ JK (Jumping Knowledge) connections capture multi-scale structure      ║
╠══════════════════════════════════════════════════════════════════════════╣
║ TRAINING                                                                 ║
║ ✓ Adam with lr=1e-4 to 3e-4, weight_decay=1e-5                         ║
║ ✓ CosineAnnealingLR or ReduceLROnPlateau                                ║
║ ✓ Early stopping on validation AUC (not loss)                           ║
║ ✓ Batch size 32–64 — larger batches → less noisy gradients              ║
║ ✓ Evaluate with ROC-AUC AND PR-AUC — PR-AUC more informative for       ║
║   imbalanced datasets (which all toxicity datasets are)                 ║
╠══════════════════════════════════════════════════════════════════════════╣
║ CHOOSING A MODEL                                                         ║
║ ✓ GCN    : fastest, great baseline, no edge features                    ║
║ ✓ GAT    : better than GCN, interpretable (attention), no edge feats    ║
║ ✓ MPNN   : best when bond type matters (most chemistry tasks)           ║
║ ✓ GIN    : best expressiveness guarantee, sum aggregation               ║
║ ✓ AFP    : SOTA for molecular property prediction, production standard  ║
║   Rule: try GCN first → AttentiveFP second → tune the winner           ║
╠══════════════════════════════════════════════════════════════════════════╣
║ DATASET SIZE GUIDELINES                                                  ║
║ < 500 compounds  → fingerprint RF/XGBoost often beats GNN               ║
║ 500–5,000        → GNN competitive, use scaffold split carefully        ║
║ > 5,000          → GNN typically wins, especially with pretraining      ║
║ > 100,000        → Graph Transformer (Grover, GraphGPS)                 ║
╠══════════════════════════════════════════════════════════════════════════╣
║ COMMON PITFALLS                                                          ║
║ ✗ Random split → 5–15% inflated AUC vs scaffold split                  ║
║ ✗ Not adding self-loops for GCN → poor performance                     ║
║ ✗ Missing BatchNorm → training diverges                                 ║
║ ✗ Using mean pool for Set2Set models → worse readout                   ║
║ ✗ Forgetting to add both directed edges → graph is disconnected        ║
║ ✗ Reporting AUC on val set (used for early stopping) as test AUC       ║
╚══════════════════════════════════════════════════════════════════════════╝
"""
print(pitfalls)

---
## What to learn next

| Topic | Resource |
|---|---|
| Multi-task GNN (Tox21 12 endpoints) | Notebook T08 in this repo |
| Pretrained GNNs (GNN + transfer) | Hu et al. 2020 (Strategies for Pre-training GNNs) |
| Graph Transformers | GPS (Rampásek 2022), Grover (Rong 2020) |
| 3D-aware GNNs | SchNet, DimeNet++, PaiNN |
| ChemProp (production MPNN) | `pip install chemprop` — AstraZeneca's tool |
| MoleculeNet benchmarks | moleculenet.org |
| PyG tutorials | pytorch-geometric.readthedocs.io |

*Built by Himanshu Goel · [himanshugoel.github.io](https://himanshugoel.github.io)*